# RAG-Based Educational Chatbot
### Learns directly from your PDF study materials

**Subjects Covered:** Mathematics | Science | English | Social Studies

---

## Task 1 — RAG Pipeline Design

The complete RAG pipeline works as follows:

```
[PDF Study Materials]        ← Your actual textbook PDFs
        ↓
[Document Loading]           ← PyMuPDF extracts text from PDFs
        ↓
[Text Chunking]              ← Split into overlapping pieces
        ↓
[Embedding Generation]       ← sentence-transformers converts chunks to vectors
        ↓
[Vector Database Storage]    ← ChromaDB stores all vectors
        ↓
[Student Question]
        ↓
[Similarity Search]          ← Find top-3 most relevant chunks
        ↓
[Prompt Engineering]         ← Inject retrieved context into prompt
        ↓
[OpenAI GPT-3.5-turbo]       ← Generate student-friendly answer
```

## Install Dependencies

In [ ]:
!pip install pymupdf chromadb sentence-transformers openai

## Task 2 — Chunking Strategy

**Chosen Strategy: Sliding Window (Fixed-size with Overlap)**

| Parameter | Value | Reason |
|-----------|-------|--------|
| Chunk size | 300 words | Captures one full concept/paragraph |
| Overlap | 50 words | Preserves context at chunk boundaries |

**Why this suits textbooks:**
- Each chunk covers one idea — matches how students search for answers
- Overlap prevents important sentences from being cut off
- Too large = noisy retrieval | Too small = incomplete answers

In [ ]:
# Task 2 — Chunking Function

def chunk_text(text, chunk_size=300, overlap=50):
    """
    Splits text into overlapping word-based chunks.
    chunk_size : number of words per chunk
    overlap    : number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunk = " ".join(words[start : start + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

print("Chunking function ready.")

## Task 3 — Embedding Model & Vector Database

**Embedding Model: `all-MiniLM-L6-v2`**
- Free, runs locally — no API key needed for embeddings
- 384-dimensional vectors with strong semantic understanding
- Captures meaning: "H₂O" and "water molecule" are close in vector space

**Vector Database: ChromaDB**
- Runs in-memory, zero server setup
- Fast cosine similarity search across thousands of chunks
- Stores subject metadata so you can filter by subject if needed

In [ ]:
# Task 3 — Load Embedding Model and Set Up ChromaDB

import chromadb
from sentence_transformers import SentenceTransformer

print("Loading embedding model (first run downloads ~90MB)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="educational_docs",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB collection ready!")

## Task 4 — Prompt Engineering

| Prompt Element | Purpose |
|---|---|
| Role: "helpful tutor" | Sets a friendly, educational tone |
| "Use ONLY the context below" | Grounds the answer in your PDFs — prevents hallucination |
| "If not in context, say so" | Honest fallback instead of making things up |
| "Simple language" | Suitable for school students |
| "Step by step if needed" | Helps explain math/science processes clearly |

In [ ]:
# Task 4 — Prompt Template

def build_prompt(context_chunks, student_question):
    context = "\n\n".join(context_chunks)
    prompt = f"""You are a helpful and friendly tutor for school students.
Answer the student's question using ONLY the context provided below.
- Use simple, easy-to-understand language.
- Explain step by step if the question involves a process or calculation.
- If the answer is not in the context, say: "I'm not sure about that — please check your textbook."
- Do NOT make up any information.

--- CONTEXT FROM TEXTBOOK ---
{context}
-----------------------------

Student's Question: {student_question}

Answer:"""
    return prompt

print("Prompt template ready.")

## Task 5 — Build the Chatbot

### Step 5a: Upload Your PDF Files

Place your PDF textbooks in a folder called `pdfs/` next to this notebook.  
Name them by subject, for example:
```
pdfs/
  science.pdf
  mathematics.pdf
  english.pdf
  social_studies.pdf
```
You can add as many PDFs as you like — all will be indexed automatically.

In [ ]:
# Step 5a — Load and Extract Text from PDFs using PyMuPDF

import fitz  # PyMuPDF
import os

PDF_FOLDER = "pdfs"  # folder containing your subject PDFs

def load_pdfs_from_folder(folder_path):
    """
    Reads all PDF files from the given folder.
    Returns a dict: { filename_without_extension : extracted_text }
    """
    documents = {}

    if not os.path.exists(folder_path):
        print(f"Folder '{folder_path}' not found. Creating it...")
        os.makedirs(folder_path)
        print(f"Please put your PDF files inside the '{folder_path}/' folder and re-run this cell.")
        return documents

    pdf_files = [f for f in os.listdir(folder_path) if f.endswith(".pdf")]

    if not pdf_files:
        print(f"No PDF files found in '{folder_path}/'. Please add your textbook PDFs there.")
        return documents

    for filename in pdf_files:
        filepath = os.path.join(folder_path, filename)
        subject_name = os.path.splitext(filename)[0]  # e.g. 'science' from 'science.pdf'

        doc = fitz.open(filepath)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        doc.close()

        documents[subject_name] = full_text
        print(f"  Loaded: {filename}  ({len(full_text.split())} words)")

    return documents


print("Loading PDFs...")
educational_documents = load_pdfs_from_folder(PDF_FOLDER)
print(f"\nTotal subjects loaded: {list(educational_documents.keys())}")

### Step 5b: Chunk, Embed, and Store in Vector DB

In [ ]:
# Step 5b — Chunk all PDF text, embed, and store in ChromaDB

all_chunks = []
all_ids = []
all_metadata = []

for subject, text in educational_documents.items():
    chunks = chunk_text(text, chunk_size=300, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{subject}_chunk_{i}")
        all_metadata.append({"subject": subject})
    print(f"  {subject}: {len(chunks)} chunks")

print(f"\nTotal chunks across all PDFs: {len(all_chunks)}")

# Generate embeddings in batches for speed
print("Generating embeddings (this may take a moment)...")
embeddings = embedding_model.encode(all_chunks, batch_size=64, show_progress_bar=True).tolist()

# Store in ChromaDB
collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    ids=all_ids,
    metadatas=all_metadata
)

print("\nAll chunks stored in vector database!")

### Step 5c: Retrieval Function

In [ ]:
# Step 5c — Retrieve the most relevant chunks for a question

def retrieve_relevant_chunks(question, top_k=3):
    """
    Converts the question to a vector and finds the top_k closest chunks.
    """
    question_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k
    )
    return results["documents"][0]  # list of top-k chunk texts


# Quick test — change this question to match your PDFs
test_q = "What is photosynthesis?"
retrieved = retrieve_relevant_chunks(test_q)
print(f"Top chunks for: '{test_q}'\n")
for i, chunk in enumerate(retrieved, 1):
    print(f"[Chunk {i}]:\n{chunk}\n{'-'*50}")

### Step 5d: Set OpenAI API Key

In [ ]:
# Step 5d — Set your OpenAI API Key
import openai

openai.api_key = "sk-..."  # ← Replace with your actual OpenAI API key

print("OpenAI client ready!")

### Step 5e: Answer Generation using OpenAI GPT

In [ ]:
# Step 5e — Full RAG pipeline: retrieve → prompt → generate answer

def generate_answer(question):
    # 1. Retrieve relevant chunks from the PDF vector store
    context_chunks = retrieve_relevant_chunks(question, top_k=3)

    # 2. Build the prompt with retrieved context
    prompt = build_prompt(context_chunks, question)

    # 3. Send to OpenAI GPT
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.3  # lower = more factual
    )

    return response.choices[0].message.content


# Test with a question from your PDFs
answer = generate_answer("What is photosynthesis?")
print(answer)

### Step 5f: Interactive Chatbot Loop

In [ ]:
# Step 5f — Chat with your textbooks!

print("=" * 55)
print("  Educational Chatbot — Powered by your PDF textbooks")
print(f"  Loaded subjects: {list(educational_documents.keys())}")
print("  Type 'quit' to exit")
print("=" * 55)

while True:
    question = input("\nYou: ").strip()

    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye! Happy studying!")
        break

    if not question:
        continue

    print("Chatbot: Searching your textbooks...")
    answer = generate_answer(question)
    print(f"\nChatbot: {answer}")
    print("-" * 55)